# Migrating PyTorch deep learning workloads to the Ascend ecosystem

[PyTorch](https://pytorch.org/) is an open source deep learning framework in Python originally developed by [Meta](https://www.meta.com/), now hosted under the vendor-neutral [PyTorch Foundation](https://pytorch.org/foundation/). It is the de-facto industry standard for training modern deep learning models and large language models \(LLMs\). Like most other deep learning frameworks, PyTorch natively supports accelerating machine learning workloads on NVIDIA GPUs through its [CUDA](https://developer.nvidia.com/cuda/toolkit) ecosystem without the need for special plugins or adapters.

The [`torch-npu`](https://pypi.org/project/torch-npu/) plugin developed by the Huawei [Ascend](https://www.hiascend.com/en) community enables AI/ML engineers to migrate machine learning workloads from NVIDIA GPUs to Ascend NPUs seamlessly with minimal changes to existing code and processes. With the high-level model training logic in PyTorch unchanged, the [CANN](https://www.hiascend.com/en/cann) kernels library is used in place of CUDA and Ascend NPUs are used for hardware acceleration in place of NVIDIA GPUs.

This notebook experiment serves as a gentle introduction to migrating deep learning models written in PyTorch to the Ascend ecosystem, using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset as our motivating example.

## Prerequisites

The content in this notebook experiment builds upon the first 6 chapters of [Dive into Deep Learning](https://d2l.ai/), also known as D2L.

## Python version and dependencies

This notebook experiment runs on Python 3.11 with the following Python package dependencies.

1. PyTorch 2.8.0
1. [PyYAML](https://pypi.org/project/PyYAML/) 6.0.3
1. [setuptools](https://pypi.org/project/setuptools/) 82.0.1
1. `torch-npu` 2.8.0
1. CANN 8.5.0

In [1]:
!cat requirements.txt

absl-py==2.4.0
attrs==25.4.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.6
jupyterlab-git==0.52.0
jupyter-resource-usage==1.2.0
ml-dtypes==0.5.4
torch==2.8.0
torch-npu==2.8.0
pyyaml==6.0.3
scipy==1.17.1
setuptools==82.0.1
sympy==1.14.0
tornado==6.5.5


In [2]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Confirming that NPU acceleration is available

The `torch.npu.is_available` method reports whether Ascend NPU acceleration is available.

In [3]:
import torch
import torch_npu

torch.npu.is_available()

/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0/aarch64-linux/ascend_toolkit_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/ora

True

Warnings can be safely ignored unless you see messages starting with `[ERROR]` or `[CRITICAL]` in which case consult the Ascend forum for assistance.

As shown in the output above, NPU acceleration is available on our [OrangePi AIpro \(20T\)](http://www.orangepi.org/html/hardWare/computerAndMicrocontrollers/details/Orange-Pi-AIpro%2820t%29.html) development board which includes a single Ascend 310B1 NPU chip and core.

We can also get the number of available NPUs with `torch.npu.device_count` and the current device ID with `torch.npu.current_device`.

In [4]:
torch.npu.device_count()

1

In [5]:
torch.npu.current_device()

0

Our OrangePi AIpro \(20T\) development board has a single NPU chip and core so PyTorch returns `1` for the device count and `0` for the current device ID as expected.

Let's verify the results from PyTorch with the `npu-smi` command-line utility.

In [6]:
!npu-smi info

+--------------------------------------------------------------------------------------------------------+
| npu-smi 23.0.0                                   Version: 23.0.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          51                24    / 24            |
| 0       0                     | NA              | 0            7606 / 23673                            |
+===============================+=================+======================================================+


As reported above, we have a single Ascend 310B1 NPU chip and core on our development board.

## Copying tensors to device memory

PyTorch tensors reside in main memory by default. Main memory is also commonly known as random access memory \(RAM\) or CPU RAM.

Let's create a $2 \times 2$ matrix with [`torch.randn`](https://docs.pytorch.org/docs/2.8/generated/torch.randn.html) and query its `device` attribute to confirm that our tensor resides in main memory.

In [7]:
A = torch.randn(2, 2)
A, A.device

(tensor([[ 0.0028,  0.3004],
         [-1.1344,  0.6903]]),
 device(type='cpu'))

The `device` attribute returns `device(type='cpu')` which confirms our tensor is residing in main memory. In many cases, we'll want to copy our tensor to NPU device memory. This allows us to perform computationally intensive operations such as matrix multiplication directly on the NPU device to speed up the model training process.

Let's use the `npu` method on our tensor to move it to \(NPU\) device memory and confirm that the `device` attribute on our tensor is updated appropriately.

In [8]:
A = A.npu()
A, A.device

[W429 20:05:40.578786747 compiler_depend.ts:164] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())


.

(tensor([[ 0.0028,  0.3004],
         [-1.1344,  0.6903]], device='npu:0'),
 device(type='npu', index=0))

The `device` attribute of our resulting tensor now reports `device(type='npu', index=0)`, confirming that it is now copied to device memory.

## Initializing tensors directly to device memory

Instead of creating our tensors in main memory only to copy them to device memory, we can initialize our tensors directly to device memory. This spares us the overhead of copying our data across different devices which poses non-trivial communication overhead.

Use `torch.device` to construct an object representing our NPU device, then specify it in the `device` parameter of tensor constructors such as `torch.randn`. Alternatively, we can specify the `device` parameter during tensor initialization as the string `'npu:0'` directly - both are functionally equivalent.

In [9]:
npu_0 = torch.device('npu:0')
npu_0

device(type='npu', index=0)

In [10]:
M0 = torch.randn(2, 2, device=npu_0)
M1 = torch.randn(2, 2, device='npu:0')
M0, M1, M0.device, M1.device

(tensor([[0.6995, 0.9877],
         [1.1796, 0.8158]], device='npu:0'),
 tensor([[-2.6261, -0.6210],
         [-2.7608, -0.3618]], device='npu:0'),
 device(type='npu', index=0),
 device(type='npu', index=0))

Let's multiply the 2 matrices with the `@` operator. As both matrices are on the same NPU device with index `0`, the product also automatically resides on the same NPU device memory.

In [11]:
M = M0 @ M1
M, M.device

(tensor([[-4.5639, -0.7918],
         [-5.3498, -1.0277]], device='npu:0'),
 device(type='npu', index=0))

## Automatic migration of PyTorch CUDA code to Ascend

Imagine you have a deep learning project written in PyTorch leveraging NVIDIA's CUDA framework for GPU acceleration. Your codebase may already be riddled with CUDA calls like such: `X.cuda()`

To migrate your codebase to run on Ascend NPUs, you would normally need to refactor the codebase manually to replace the CUDA calls to `npu` calls: `X.cuda()` $\rightarrow$ `X.npu()`. While this in itself is relatively straightforward, it is nevertheless grunt work producing no business value which you would rather avoid.

Fortunately, the `torch_npu.contrib` module provides the `transfer_to_npu` submodule which you can import by adding a single line to your existing codebase. It automatically intercepts CUDA calls and converts them to NPU calls under the hood.

Let's see it in action!

In [12]:
from torch_npu.contrib import transfer_to_npu

torch.cuda.is_available()

/home/HwHiAiUser/.pyenv/versions/3.11.15/envs/orangepiaipro-20t/lib/python3.11/site-packages/torch_npu/contrib/transfer_to_npu.py:347: ImportWarning: 
    *************************************************************************************************************
    The torch.Tensor.cuda and torch.nn.Module.cuda are replaced with torch.Tensor.npu and torch.nn.Module.npu now..
    The torch.cuda.DoubleTensor is replaced with torch.npu.FloatTensor cause the double type is not supported now..
    The backend in torch.distributed.init_process_group set to hccl now..
    The torch.cuda.* and torch.cuda.amp.* are replaced with torch.npu.* and torch.npu.amp.* now..
    The device parameters have been replaced with npu in the function below:
    torch.logspace, torch.randint, torch.hann_window, torch.rand, torch.full_like, torch.ones_like, torch.rand_like, torch.randperm, torch.arange, torch.frombuffer, torch.normal, torch._empty_per_channel_affine_quantized, torch.empty_strided, torch.empty

True

In [13]:
torch.cuda.device_count(), torch.cuda.current_device()

(1, 0)

In [14]:
gpu_0 = torch.device('cuda:0')
gpu_0

device(type='cuda', index=0)

In [15]:
P0 = torch.randn(2, 2, device=gpu_0)
P1 = torch.randn(2, 2, device='cuda:0')
P0, P1, P0.device, P1.device

(tensor([[-0.7419,  0.1012],
         [ 0.2954, -1.4987]], device='npu:0'),
 tensor([[2.2383, 1.4669],
         [0.3825, 0.3697]], device='npu:0'),
 device(type='npu', index=0),
 device(type='npu', index=0))

In [16]:
P = P0 @ P1
P, P.device

(tensor([[-1.6218, -1.0508],
         [ 0.0880, -0.1208]], device='npu:0'),
 device(type='npu', index=0))

With our PyTorch CUDA codebase seamlessly migrated to Ascend NPU, let's see a more realistic example in action - training a multilayer perceptron \(MLP\) model on the Fashion MNIST dataset. We'll use CUDA calls throughout our motivating example to demonstrate that none of your existing code needs to be manually migrated - what worked on NVIDIA GPUs will continue to work on Ascend NPUs as-is!

## Training a deep neural network on the Fashion MNIST dataset

TODO